In [1]:
# Este comando descarga el repositorio entero a una carpeta llamada 'TFMDS' en Colab.
#!git clone https://github.com/jmorala/TFMDS.git

# Inicializar directorios
Clonar repositorio github
Posicionarse en el directorio raíz

In [2]:
import os
import sys

# ============================================================================
# CONFIGURACIÓN DE DIRECTORIOS
# ============================================================================

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    project_dir = '/content/TFMDS'
    os.chdir(project_dir)
else:
    project_dir = r'C:\Users\jmora\Documents\TFMDS'
    os.chdir(project_dir)

# Agregar el directorio del proyecto al path de Python
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print("Directorio de trabajo:", os.getcwd())
print("Python path incluye proyecto:", project_dir in sys.path)

Directorio de trabajo: C:\Users\jmora\Documents\TFMDS
Python path incluye proyecto: True


In [3]:
# ============================================================================
# IMPORTACIONES BASE Y VERSIÓN
# ============================================================================
import os, sys
import pandas as pd
import numpy as np
import time

import torch
import pytorch_lightning as pl
from neuralforecast import NeuralForecast, __version__ as nf_version
from neuralforecast.models import LSTM
from neuralforecast.losses.pytorch import MAE

# Importar utilidades desde la carpeta 'lib' evitando el paquete
lib_dir = os.path.join(os.getcwd(), 'lib')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

from dl_utils import (
    preparar_datos_neuralforecast,
    seleccionar_features_exogenas,
    reconstruir_predicciones,
)
from metricas import calcular_metricas, resumen_metricas
from graficos_dl import (
    grafico_prediccion_diaria_agregada,
    grafico_prediccion_por_cluster,
    grafico_productos_por_cluster,
    dashboard_metricas_dl,
)

print(f"torch: {torch.__version__}")
print(f"pytorch_lightning: {pl.__version__}")
print(f"neuralforecast: {nf_version}")


c:\Users\jmora\Documents\TFMDS\venv\Lib\site-packages\lightning_utilities\core\imports.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


torch: 2.9.1+cpu
pytorch_lightning: 2.0.9
neuralforecast: 1.6.4


# Lectura de datos preparados para Deep Learning

In [4]:
# ============================================================================
# LECTURA DE DATOS
# ============================================================================

print("\n" + "="*100)
print("📂 CARGANDO DATOS PARA DEEP LEARNING")
print("="*100)

df_train_raw = pd.read_csv('datos/df_train_dl.csv', sep=';', parse_dates=['idSecuencia'])
df_test_raw = pd.read_csv('datos/df_test_dl.csv', sep=';', parse_dates=['idSecuencia'])

print(f"\n✅ Datos cargados:")
print(f"   Train: {df_train_raw.shape}")
print(f"   Test:  {df_test_raw.shape}")
print(f"\n📋 Columnas: {list(df_train_raw.columns)}")


📂 CARGANDO DATOS PARA DEEP LEARNING

✅ Datos cargados:
   Train: (625800, 27)
   Test:  (27714, 27)

📋 Columnas: ['idSecuencia', 'producto', 'udsVenta', 'bolPromocion', 'bolOpen', 'bolHoliday', 'lag_ventas_1', 'lag_ventas_2', 'lag_ventas_3', 'lag_ventas_4', 'lag_ventas_5', 'lag_ventas_6', 'lag_ventas_7', 'media_mes_anterior', 'EWMA_corto', 'EWMA_largo', 'Tendencia_EWMA', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos', 'trimestre_sin', 'trimestre_cos', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3']

✅ Datos cargados:
   Train: (625800, 27)
   Test:  (27714, 27)

📋 Columnas: ['idSecuencia', 'producto', 'udsVenta', 'bolPromocion', 'bolOpen', 'bolHoliday', 'lag_ventas_1', 'lag_ventas_2', 'lag_ventas_3', 'lag_ventas_4', 'lag_ventas_5', 'lag_ventas_6', 'lag_ventas_7', 'media_mes_anterior', 'EWMA_corto', 'EWMA_largo', 'Tendencia_EWMA', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos', 'trimestre_sin', 'trimestre_cos', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3']

# Preparación de datos para NeuralForecast

In [ ]:
# ============================================================================
# PREPARACIÓN DE DATOS PARA NEURALFORECAST
# ============================================================================

print("\n" + "="*100)
print("🔧 PREPARANDO DATOS PARA NEURALFORECAST")
print("="*100)

# Convertir al formato NeuralForecast (unique_id, ds, y)
df_train_nf, df_test_nf = preparar_datos_neuralforecast(
    df_train_raw,
    df_test_raw,
    col_fecha='idSecuencia',
    col_producto='producto',
    col_target='udsVenta'
)

# Identificar features exógenas
exog_features = seleccionar_features_exogenas(
    df_train_nf,
    excluir=['producto_encoded']  # Excluir columnas redundantes
)

print(f"\n📊 Features exógenas para el modelo: {len(exog_features)}")
print(f"   {exog_features}")

# Configuración del modelo LSTM

In [ ]:
# ============================================================================
# CONFIGURACIÓN DEL MODELO LSTM
# ============================================================================

print("\n" + "="*100)
print("🧠 CONFIGURANDO MODELO LSTM")
print("="*100)

# Horizonte de predicción (30 días)
HORIZON = 30

# Hiperparámetros del modelo LSTM
modelo_lstm = LSTM(
    h=HORIZON,                      # Horizonte de predicción
    input_size=60,                  # Ventana de entrada (últimos 60 días)
    loss=MAE(),                     # Función de pérdida
    max_steps=500,                  # Número máximo de pasos de entrenamiento
    early_stop_patience_steps=50,   # Early stopping
    encoder_hidden_size=128,        # Tamaño capa oculta LSTM
    encoder_n_layers=2,             # Número de capas LSTM
    context_size=10,                # Contexto adicional
    decoder_hidden_size=128,        # Tamaño decoder
    decoder_layers=2,               # Capas decoder
    learning_rate=1e-3,             # Learning rate
    scaler_type='robust',           # Escalado robusto
    batch_size=32,                  # Batch size
    random_seed=42,                 # Semilla aleatoria
    futr_exog_list=exog_features,   # Variables exógenas futuras
    trainer_kwargs={                # Forzar ejecución en CPU y sin distribuido
        'accelerator': 'cpu',
        'devices': 1,
        'enable_progress_bar': True
    }
)

print("\n✅ Modelo LSTM configurado:")
print(f"   Horizonte: {HORIZON} días")
print(f"   Input size: {60} días")
print(f"   Hidden size: {128}")
print(f"   Layers: {2}")
print(f"   Max steps: {500}")
print(f"   Features exógenas: {len(exog_features)}")

# Entrenamiento del modelo

In [ ]:
# ============================================================================
# ENTRENAMIENTO DEL MODELO
# ============================================================================

print("\n" + "="*100)
print("🚀 INICIANDO ENTRENAMIENTO LSTM")
print("="*100)

# Crear instancia de NeuralForecast
nf = NeuralForecast(
    models=[modelo_lstm],
    freq='D'  # Frecuencia diaria
)

# Entrenar el modelo
print("\n⏳ Entrenando modelo LSTM...")
start_time = time.perf_counter()

nf.fit(df=df_train_nf)

elapsed = time.perf_counter() - start_time

print(f"\n✅ Entrenamiento completado en {elapsed:.2f} segundos ({elapsed/60:.2f} minutos)")

# Predicción sobre el conjunto de test

In [ ]:
# ============================================================================
# PREDICCIÓN
# ============================================================================

print("\n" + "="*100)
print("🔮 GENERANDO PREDICCIONES")
print("="*100)

# Realizar predicciones
print("\n⏳ Generando predicciones en test set...")
start_time = time.perf_counter()

y_hat = nf.predict(futr_df=df_test_nf)

elapsed = time.perf_counter() - start_time

print(f"\n✅ Predicciones generadas en {elapsed:.2f} segundos")
print(f"   Shape predicciones: {y_hat.shape}")
print(f"\n📋 Primeras predicciones:")
print(y_hat.head())

# Reconstruir predicciones en formato original

In [ ]:
# ============================================================================
# RECONSTRUIR PREDICCIONES
# ============================================================================

print("\n" + "="*100)
print("🔧 RECONSTRUYENDO PREDICCIONES EN FORMATO ORIGINAL")
print("="*100)

# Reconstruir con columnas originales
df_test_pred = reconstruir_predicciones(
    y_hat=y_hat,
    df_test_original=df_test_raw,
    modelo_name='LSTM',
    col_producto='producto',
    col_fecha='idSecuencia'
)

# Calcular errores
df_test_pred['error'] = df_test_pred['prediccion'] - df_test_pred['udsVenta']
df_test_pred['error_abs'] = np.abs(df_test_pred['error'])

print(f"\n✅ Dataset reconstruido:")
print(f"   Shape: {df_test_pred.shape}")
print(f"   Columnas: {list(df_test_pred.columns)}")
print(f"\n📊 Estadísticas básicas del error:")
print(f"   Error medio: {df_test_pred['error'].mean():.2f}")
print(f"   Error abs medio: {df_test_pred['error_abs'].mean():.2f}")
print(f"   Error std: {df_test_pred['error'].std():.2f}")

# Cálculo de métricas

In [ ]:
# ============================================================================
# CÁLCULO DE MÉTRICAS
# ============================================================================

print("\n" + "="*100)
print("📊 CÁLCULO DE MÉTRICAS")
print("="*100)

# Filtrar valores válidos (sin NaN)
df_valid = df_test_pred.dropna(subset=['prediccion', 'udsVenta'])

# Calcular métricas
metricas_lstm = calcular_metricas(
    y=df_valid['udsVenta'],
    y_pred=df_valid['prediccion'],
    name='LSTM'
)

# Mostrar resumen
resumen_metricas([metricas_lstm])

# Guardar para comparación posterior
todas_metricas = [metricas_lstm]

# Visualizaciones

## 1. Predicción diaria agregada (todos los productos)

In [ ]:
grafico_prediccion_diaria_agregada(
    df=df_test_pred,
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    titulo='LSTM - Ventas Diarias Agregadas (Todos los Productos)',
    figsize=(14, 5)
)

## 2. Predicciones por Cluster

In [ ]:
grafico_prediccion_por_cluster(
    df=df_test_pred,
    col_cluster='Cluster',
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    figsize=(16, 10)
)

## 3. Top 2 productos por Cluster

In [ ]:
grafico_productos_por_cluster(
    df=df_test_pred,
    col_cluster='Cluster',
    col_producto='producto',
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    n_productos_por_cluster=2,
    figsize=(18, 12)
)

## 4. Dashboard de métricas

In [ ]:
# Convertir lista de métricas a diccionario
metricas_dict = {m['Algoritmo']: m for m in todas_metricas}

dashboard_metricas_dl(
    metricas_dict=metricas_dict,
    titulo='Dashboard de Métricas - LSTM',
    figsize=(16, 10)
)

# Guardar resultados